# Capturing changes with triggers, and consuming them

This notebook runs the database blocks of the lesson *Build a small streaming ingestion pipeline*. It uses only the `sqlite3` module of the standard library, with a database in memory, so nothing has to be installed and no key is needed. The outputs saved in the cells come from a real run. It shows trigger-based capture, not log-based capture.

## 1. Tables and triggers

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE article (id TEXT PRIMARY KEY, body TEXT, version INTEGER);
CREATE TABLE changes (seq INTEGER PRIMARY KEY AUTOINCREMENT,
                      article_id TEXT, op TEXT, version INTEGER, body TEXT);
CREATE TRIGGER on_insert AFTER INSERT ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (NEW.id, 'upsert', NEW.version, NEW.body);
END;
CREATE TRIGGER on_update AFTER UPDATE ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (NEW.id, 'upsert', NEW.version, NEW.body);
END;
CREATE TRIGGER on_delete AFTER DELETE ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (OLD.id, 'delete', OLD.version + 1, NULL);
END;
""")

query = "SELECT type, name FROM sqlite_master WHERE name NOT LIKE 'sqlite_%' ORDER BY type, name"
for kind, name in db.execute(query):
    print(kind, name)

table article
table changes
trigger on_delete
trigger on_insert
trigger on_update


## 2. What the triggers record

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE article (id TEXT PRIMARY KEY, body TEXT, version INTEGER);
CREATE TABLE changes (seq INTEGER PRIMARY KEY AUTOINCREMENT,
                      article_id TEXT, op TEXT, version INTEGER, body TEXT);
CREATE TRIGGER on_insert AFTER INSERT ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (NEW.id, 'upsert', NEW.version, NEW.body);
END;
CREATE TRIGGER on_update AFTER UPDATE ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (NEW.id, 'upsert', NEW.version, NEW.body);
END;
CREATE TRIGGER on_delete AFTER DELETE ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (OLD.id, 'delete', OLD.version + 1, NULL);
END;
""")

with db:                                                 # one transaction: commit at the end
    db.execute("INSERT INTO article VALUES ('refunds', 'Paid in 10 days.', 1)")
    db.execute("INSERT INTO article VALUES ('gift-cards', 'Valid 12 months.', 1)")
with db:
    db.execute("UPDATE article SET body = 'Paid in 5 days.', version = 2 WHERE id = 'refunds'")
try:
    with db:                                             # this transaction is rolled back
        db.execute("UPDATE article SET body = 'Paid in 1 day.', version = 3 WHERE id = 'refunds'")
        raise RuntimeError("the save failed")
except RuntimeError:
    pass
with db:
    db.execute("DELETE FROM article WHERE id = 'gift-cards'")

for row in db.execute("SELECT seq, article_id, op, version, body FROM changes ORDER BY seq"):
    print(row)

(1, 'refunds', 'upsert', 1, 'Paid in 10 days.')
(2, 'gift-cards', 'upsert', 1, 'Valid 12 months.')
(3, 'refunds', 'upsert', 2, 'Paid in 5 days.')
(4, 'gift-cards', 'delete', 2, None)


## 3. A consumer that polls the change table

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE article (id TEXT PRIMARY KEY, body TEXT, version INTEGER);
CREATE TABLE changes (seq INTEGER PRIMARY KEY AUTOINCREMENT,
                      article_id TEXT, op TEXT, version INTEGER, body TEXT);
CREATE TRIGGER on_insert AFTER INSERT ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (NEW.id, 'upsert', NEW.version, NEW.body);
END;
CREATE TRIGGER on_update AFTER UPDATE ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (NEW.id, 'upsert', NEW.version, NEW.body);
END;
CREATE TRIGGER on_delete AFTER DELETE ON article BEGIN
  INSERT INTO changes (article_id, op, version, body)
  VALUES (OLD.id, 'delete', OLD.version + 1, NULL);
END;
""")

with db:                                                 # one transaction: commit at the end
    db.execute("INSERT INTO article VALUES ('refunds', 'Paid in 10 days.', 1)")
    db.execute("INSERT INTO article VALUES ('gift-cards', 'Valid 12 months.', 1)")
with db:
    db.execute("UPDATE article SET body = 'Paid in 5 days.', version = 2 WHERE id = 'refunds'")
try:
    with db:                                             # this transaction is rolled back
        db.execute("UPDATE article SET body = 'Paid in 1 day.', version = 3 WHERE id = 'refunds'")
        raise RuntimeError("the save failed")
except RuntimeError:
    pass
with db:
    db.execute("DELETE FROM article WHERE id = 'gift-cards'")

index, last_seq = {}, 0                                  # key -> (version, body or None)

def consume(limit=100):
    global last_seq
    rows = db.execute("SELECT seq, article_id, op, version, body FROM changes "
                      "WHERE seq > ? ORDER BY seq LIMIT ?", (last_seq, limit)).fetchall()
    for seq, key, op, version, body in rows:
        if version > index.get(key, (0, None))[0]:
            index[key] = (version, body)
        last_seq = seq                                   # moved only after the index write
    return len(rows)

print("first batch:", consume(limit=2), "| last_seq", last_seq)
print("second batch:", consume(), "| last_seq", last_seq)
print("nothing new:", consume())
before = dict(index)
last_seq = 0                                             # replay everything
print("replayed:", consume(), "| index unchanged:", index == before)
print(index)

first batch: 2 | last_seq 2
second batch: 2 | last_seq 4
nothing new: 0
replayed: 4 | index unchanged: True
{'refunds': (2, 'Paid in 5 days.'), 'gift-cards': (2, None)}


## Try next

Store `last_seq` in a table and write it in the same transaction as the index rows. Use invented data only.